# 🧼 Clean Data + Build 5 Data Marts

Notebook chạy `src/clean_data.py` + `src/build_marts.py` tạo 5 Marts cover 13/13 bảng raw.

**5 Marts**:
| Mart | Grain | Use case |
|------|-------|----------|
| 1. order_lines | 1 = 1 orderline | EDA chéo nhiều chiều |
| 2. orders | 1 = 1 order | Customer behavior |
| 3. products | 1 = 1 product | Product diagnostics |
| 4. daily_ops | 1 = 1 day | Forecasting + temporal trends (đã gộp date_dim) |
| 5. promotion_perf | 1 = 1 promo | Promo ROI |

**Prerequisites**:
- Đã chạy `01_data_profiling.ipynb`.
- 13 file CSV trong `data/raw/`.
- Đã `pip install pyarrow` (cho parquet).

**Output**:
- `data/interim/` : 13 file parquet (đã clean) + `clean_report.json`
- `data/processed/` : 5 file parquet (Marts, zstd compressed)

## Cell 0 — Setup

In [1]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(f'CWD: {os.getcwd()}')
print(f'data/raw exists: {Path("data/raw").exists()}')
print(f'src/ exists: {Path("src").exists()}')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

CWD: C:\Users\admin\Desktop\The-Richers
data/raw exists: True
src/ exists: True


## Cell 1 — Chạy clean_data

In [2]:
from src.clean_data import clean_all
cleaned = clean_all(save=True)
print(f'\n✓ Cleaned {len(cleaned)} tables')

🧼 CLEANING PIPELINE
🔄 Loading raw CSVs...
  ✓ products        shape=(2412, 8)
  ✓ customers       shape=(121930, 7)
  ✓ promotions      shape=(50, 10)
  ✓ geography       shape=(39948, 4)
  ✓ orders          shape=(646945, 8)
  ✓ order_items     shape=(714669, 7)
  ✓ payments        shape=(646945, 4)
  ✓ shipments       shape=(566067, 4)
  ✓ returns         shape=(39939, 7)
  ✓ reviews         shape=(113551, 7)
  ✓ sales_train     shape=(3833, 3)
  ✓ inventory       shape=(60247, 17)
  ✓ web_traffic     shape=(3652, 7)

[1/3] Master tables...

[2/3] Transaction tables...

[3/3] Analytical & operational...

💾 Saving to data/interim/ as parquet (zstd)...
  ✓ products        →    2,412 rows
  ✓ customers       →  121,930 rows
  ✓ promotions      →       50 rows
  ✓ geography       →   39,948 rows
  ✓ orders          →  646,945 rows
  ✓ order_items     →  714,669 rows
  ✓ payments        →  646,945 rows
  ✓ shipments       →  566,067 rows
  ✓ returns         →   39,939 rows
  ✓ reviews    

## Cell 2 — Đọc clean_report.json

In [3]:
import json
with open('data/interim/clean_report.json', encoding='utf-8') as f:
    report = json.load(f)
rows = []
for table, actions in report.items():
    for action, info in actions.items():
        rows.append({'table': table, 'action': action, **info})
df_report = pd.DataFrame(rows)
df_report.to_csv('data/interim/clean_report.csv', index=False)
df_report

,table,action,n_before,n_after,n_dropped,note
0,products,00_loaded,2412,2412,0,"shape=(2412, 8)"
1,products,01_add_gross_margin,2412,2412,0,gross_margin = (price-cogs)/price
2,customers,00_loaded,121930,121930,0,"shape=(121930, 7)"
3,promotions,00_loaded,50,50,0,"shape=(50, 10)"
4,promotions,01_add_duration,50,50,0,promo_duration_days = end_date - start_date
5,geography,00_loaded,39948,39948,0,"shape=(39948, 4)"
6,orders,00_loaded,646945,646945,0,"shape=(646945, 8)"
7,orders,01_flag_order_before_signup,646945,646945,0,"477,453 đơn (~73.8%) flagged"
8,orders,02_add_revenue_eligible,646945,646945,0,"580,208 đơn revenue-eligible"
9,order_items,00_loaded,714669,714669,0,"shape=(714669, 7)"


## Cell 3 — Sanity check sau clean

In [4]:
# Q5 sanity
oi = pd.read_parquet('data/interim/order_items.parquet')
pct = oi['promo_id'].notna().mean() * 100
print(f'Q5 — % rows promo_id: {pct:.2f}%')
assert 38 < pct < 40
assert 'promo_id_2' not in oi.columns

# Q9 sanity
products = pd.read_parquet('data/interim/products.parquet')
returns = pd.read_parquet('data/interim/returns.parquet')
oi_size = oi.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ratio = (ret_size.groupby('size').size() / oi_size.groupby('size').size()).sort_values(ascending=False)
print(f'Q9 — size cao nhất: {ratio.idxmax()!r}')
assert ratio.idxmax() == 'S'

# Promotions có promo_duration_days
promo = pd.read_parquet('data/interim/promotions.parquet')
assert 'promo_duration_days' in promo.columns
print(f'Promotions duration range: {promo["promo_duration_days"].min()}-{promo["promo_duration_days"].max()} days')

# Order_items có discount_pct
assert 'discount_pct' in oi.columns
print(f'Discount pct range: {oi["discount_pct"].min():.4f} - {oi["discount_pct"].max():.4f}')

# Orders có cờ
orders = pd.read_parquet('data/interim/orders.parquet')
assert 'flag_order_before_signup' in orders.columns
assert 'is_revenue_eligible' in orders.columns
n_flag = orders['flag_order_before_signup'].sum()
print(f'flag_order_before_signup: {n_flag:,} ({n_flag/len(orders)*100:.1f}%)')

print('\n✅ Sanity checks PASSED')

Q5 — % rows promo_id: 38.66%
Q9 — size cao nhất: 'S'
Promotions duration range: 29-45 days
Discount pct range: 0.0000 - 0.2000
flag_order_before_signup: 477,453 (73.8%)

✅ Sanity checks PASSED


## Cell 4 — Build 5 Marts

In [5]:
from src.build_marts import build_all_marts
marts = build_all_marts(save=True)
print(f'\n✓ Built {len(marts)} marts')

🏗️  BUILDING 5 DATA MARTS
🔄 Loading cleaned tables...
  ✓ products        shape=(2412, 9)
  ✓ customers       shape=(121930, 7)
  ✓ promotions      shape=(50, 11)
  ✓ geography       shape=(39948, 4)
  ✓ orders          shape=(646945, 10)
  ✓ order_items     shape=(714669, 8)
  ✓ payments        shape=(646945, 4)
  ✓ shipments       shape=(566067, 5)
  ✓ returns         shape=(39939, 7)
  ✓ reviews         shape=(113551, 7)
  ✓ sales_train     shape=(3833, 3)
  ✓ inventory       shape=(60247, 14)
  ✓ web_traffic     shape=(3652, 7)

📦 Building MART 1: order_lines (full enrich)...
   • Start: 714,669 orderlines
   • + products       → 714,669 rows
   • + orders         → 714,669 rows
   • + customers      → 714,669 rows
   • + geography      → 714,669 rows
   • + promotions     → 714,669 rows
   ✓ Row count preserved: 714,669
   ✓ MART 1 final shape: (714669, 42)

📦 Building MART 2: orders (customer behavior)...
   • order_items agg → 646,945 orders
   • returns agg     → 36,062 orders 

## Cell 5 — Verify Mart 1: order_lines (CRITICAL)

Test integrity: row count phải đúng = order_items, tổng revenue phải khớp.

In [6]:
m1 = pd.read_parquet('data/processed/mart1_order_lines.parquet')
oi = pd.read_parquet('data/interim/order_items.parquet')

print(f'Mart 1 shape: {m1.shape}')
print(f'Cardinality preserved: {len(m1) == len(oi)} ({len(m1):,} = {len(oi):,})')

# Tổng revenue phải khớp
rev_m1 = m1['line_revenue'].sum()
rev_oi = oi['line_revenue'].sum()
print(f'\nTotal line_revenue:')
print(f'  Mart 1: {rev_m1:,.2f}')
print(f'  OI:     {rev_oi:,.2f}')
assert abs(rev_m1 - rev_oi) < 1

# Demo use case: revenue theo region × segment (1 dòng code)
print('\n=== Revenue theo region × segment (top 10) ===')
pivot = m1.groupby(['region', 'segment'])['line_revenue'].sum().sort_values(ascending=False).head(10)
print(pivot.apply(lambda x: f'{x:,.0f}').to_string())

Mart 1 shape: (714669, 42)
Cardinality preserved: True (714,669 = 714,669)

Total line_revenue:
  Mart 1: 15,680,869,265.43
  OI:     15,680,869,265.43

=== Revenue theo region × segment (top 10) ===
region   segment    
East     Everyday       2,556,568,098
         Balanced       2,323,329,743
Central  Balanced       1,688,265,206
         Everyday       1,679,476,554
East     Performance    1,145,261,713
West     Everyday         911,410,265
         Balanced         888,723,017
         Activewear       792,245,117
East     Activewear       692,256,034
West     Performance      591,953,280


## Cell 6 — Verify Mart 2: orders

In [7]:
m2 = pd.read_parquet('data/processed/mart2_orders.parquet')
print(f'Shape: {m2.shape}')
print(f'Unique order_id == n_rows: {m2["order_id"].nunique() == len(m2)}')

print(f'\nFlags & cờ:')
for col in ['flag_order_before_signup', 'flag_status_shipment_mismatch',
            'is_revenue_eligible', 'has_shipment', 'is_returned',
            'is_cancelled', 'is_reviewed']:
    n = m2[col].sum()
    print(f'  {col:35s}: {n:>8,} ({n/len(m2)*100:5.1f}%)')

print(f'\n=== Doanh thu TB / đơn theo region (revenue_eligible only) ===')
eligible = m2[m2['is_revenue_eligible'] == 1]
print(eligible.groupby('region')['order_revenue'].agg(['mean', 'sum', 'count']).round(2))

Shape: (646945, 44)
Unique order_id == n_rows: True

Flags & cờ:
  flag_order_before_signup           :  477,453 ( 73.8%)
  flag_status_shipment_mismatch      :      564 (  0.1%)
  is_revenue_eligible                :  580,208 ( 89.7%)
  has_shipment                       :  566,067 ( 87.5%)
  is_returned                        :   36,142 (  5.6%)
  is_cancelled                       :   59,462 (  9.2%)
  is_reviewed                        :  111,369 ( 17.2%)

=== Doanh thu TB / đơn theo region (revenue_eligible only) ===
             mean           sum   count
region                                 
Central  25528.04  4.224277e+09  165476
East     24723.68  6.531155e+09  264166
West     21894.64  3.296589e+09  150566


## Cell 7 — Verify Mart 3: products

In [8]:
m3 = pd.read_parquet('data/processed/mart3_products.parquet')
print(f'Shape: {m3.shape}')
print(f'Products chưa từng bán: {m3["flag_never_sold"].sum():,}')

print(f'\n=== Top 10 sản phẩm doanh thu cao nhất ===')
top10 = m3.nlargest(10, 'total_revenue')[
    ['product_name', 'category', 'segment', 'total_units_sold',
     'total_revenue', 'return_rate_qty', 'avg_rating']
]
print(top10.to_string(index=False))

print(f'\n=== Return rate TB theo segment ===')
sold = m3[m3['total_units_sold'] > 0]
print(sold.groupby('segment')['return_rate_qty'].mean().sort_values(ascending=False).round(4))

Shape: (2412, 34)
Products chưa từng bán: 814

=== Top 10 sản phẩm doanh thu cao nhất ===
     product_name   category     segment  total_units_sold  total_revenue  return_rate_qty  avg_rating
 SaigonFlex UM-92 Streetwear    Balanced           33277.0   380468714.41         0.033927    3.928571
HanoiStreet UM-10 Streetwear    Balanced           28993.0   327509101.46         0.034353    3.823869
 SaigonFlex UM-43 Streetwear    Balanced           31471.0   325014877.63         0.030854    3.894599
 SaigonFlex UM-80 Streetwear    Balanced           22709.0   256552098.24         0.032278    3.988764
 SaigonFlex UM-96 Streetwear    Balanced           24485.0   241000085.34         0.030794    3.961859
 SaigonFlex UC-69 Streetwear    Everyday           36515.0   198996641.52         0.032014    3.997801
 SaigonFlex UM-11 Streetwear    Balanced           14249.0   192410257.73         0.028493    3.955513
    UrbanVN UE-05 Streetwear Performance           35844.0   177036251.49         0.03

## Cell 8 — Verify Mart 4: daily_ops + date_dim

In [9]:
m4 = pd.read_parquet('data/processed/mart4_daily_ops.parquet')
print(f'Shape: {m4.shape}')
print(f'Range: {m4["date"].min().date()} → {m4["date"].max().date()}')
print(f'Train period rows: {m4["is_train_period"].sum():,}')

# Sanity: tổng Revenue phải = sales raw
sales_raw = pd.read_csv('data/raw/sales.csv', parse_dates=['Date'])
rev_m4 = m4[m4['is_train_period']==1]['Revenue'].sum()
rev_raw = sales_raw['Revenue'].sum()
print(f'\nSanity Revenue: mart={rev_m4:,.0f} vs raw={rev_raw:,.0f}')
assert abs(rev_m4 - rev_raw) < 1

# Test date_dim features
print(f'\n=== Tet 2023 (22/01) ===')
tet_check = m4[m4['date'].between('2023-01-15', '2023-01-25')][
    ['date', 'is_tet', 'days_to_tet', 'is_pre_tet_1w', 'is_pre_tet_2w']
]
print(tet_check.to_string(index=False))

# Pre-tet vs non-pre-tet revenue
train = m4[m4['is_train_period'] == 1].dropna(subset=['Revenue'])
print(f'\nMean Revenue khi:')
print(f'  is_pre_tet_1w=1 : {train[train["is_pre_tet_1w"]==1]["Revenue"].mean():>15,.0f}')
print(f'  is_pre_tet_1w=0 : {train[train["is_pre_tet_1w"]==0]["Revenue"].mean():>15,.0f}')
print(f'  is_1111=1       : {train[train["is_1111"]==1]["Revenue"].mean():>15,.0f}')
print(f'  is_1212=1       : {train[train["is_1212"]==1]["Revenue"].mean():>15,.0f}')
print(f'  is_weekend=1    : {train[train["is_weekend"]==1]["Revenue"].mean():>15,.0f}')
print(f'  is_weekend=0    : {train[train["is_weekend"]==0]["Revenue"].mean():>15,.0f}')

Shape: (4381, 54)
Range: 2012-07-04 → 2024-07-01
Train period rows: 3,833

Sanity Revenue: mart=16,430,476,586 vs raw=16,430,476,586

=== Tet 2023 (22/01) ===
      date  is_tet  days_to_tet  is_pre_tet_1w  is_pre_tet_2w
2023-01-15       0            7              1              1
2023-01-16       0            6              1              1
2023-01-17       0            5              1              1
2023-01-18       0            4              1              1
2023-01-19       0            3              1              1
2023-01-20       0            2              1              1
2023-01-21       0            1              1              1
2023-01-22       1            0              1              1
2023-01-23       0          383              0              0
2023-01-24       0          382              0              0
2023-01-25       0          381              0              0

Mean Revenue khi:
  is_pre_tet_1w=1 :       2,984,510
  is_pre_tet_1w=0 :       4,314,339
  is_1

## Cell 9 — Verify Mart 5: promotion_performance

In [10]:
m5 = pd.read_parquet('data/processed/mart5_promotion_perf.parquet')
print(f'Shape: {m5.shape}')
print(f'Promo chưa từng dùng: {m5["flag_unused_promo"].sum():,}/{len(m5):,}')

print(f'\n=== Top 10 promo theo doanh thu ===')
top10 = m5.nlargest(10, 'total_revenue_with_promo')[
    ['promo_id', 'promo_name', 'promo_type', 'discount_value',
     'n_orderlines_applied', 'total_revenue_with_promo',
     'revenue_per_discount_unit']
]
print(top10.to_string(index=False))

print(f'\n=== Promo theo type ===')
print(m5.groupby('promo_type').agg(
    n_promos=('promo_id', 'count'),
    total_revenue=('total_revenue_with_promo', 'sum'),
    avg_orderlines=('n_orderlines_applied', 'mean'),
).round(2))

Shape: (50, 21)
Promo chưa từng dùng: 0/50

=== Top 10 promo theo doanh thu ===
  promo_id         promo_name promo_type  discount_value  n_orderlines_applied  total_revenue_with_promo  revenue_per_discount_unit
PROMO-0021   Spring Sale 2017 percentage            12.0                  8966              179786034.74                   7.333333
PROMO-0011   Spring Sale 2015 percentage            12.0                  9594              176221955.01                   7.333333
PROMO-0017   Spring Sale 2016 percentage            12.0                  8808              171564977.24                   7.333333
PROMO-0007   Spring Sale 2014 percentage            12.0                  9373              162650369.75                   7.333333
PROMO-0027   Spring Sale 2018 percentage            12.0                  7518              148956627.41                   7.333333
PROMO-0001   Spring Sale 2013 percentage            12.0                  8523              142615559.51                   7.333

## Cell 10 — Coverage check (verify 13/13 bảng)

In [11]:
coverage = {
    'products':     ['Mart 1 (direct join)', 'Mart 3 (master)', 'Mart 5 (indirect via order_items)'],
    'customers':    ['Mart 1 (direct join)', 'Mart 2 (direct join)'],
    'promotions':   ['Mart 1 (direct join)', 'Mart 5 (master)'],
    'geography':    ['Mart 1 (direct join)', 'Mart 2 (direct join)'],
    'orders':       ['Mart 1 (direct join)', 'Mart 2 (master)', 'Mart 4 (agg daily)'],
    'order_items':  ['Mart 1 (master)', 'Mart 2 (agg)', 'Mart 3 (agg)', 'Mart 5 (agg)'],
    'payments':     ['Mart 2 (direct join)'],
    'shipments':    ['Mart 2 (direct join)'],
    'returns':      ['Mart 2 (agg)', 'Mart 3 (agg)', 'Mart 4 (agg daily)'],
    'reviews':      ['Mart 2 (agg)', 'Mart 3 (agg)'],
    'sales_train':  ['Mart 4 (master)'],
    'inventory':    ['Mart 3 (agg lifetime)', 'Mart 4 (broadcast monthly)'],
    'web_traffic':  ['Mart 4 (direct)'],
}
print('Coverage check (13/13 bảng raw → marts):')
for table, marts in coverage.items():
    print(f'  {table:15s} → {marts}')
print(f'\n✓ Tất cả 13 bảng đều xuất hiện trong ít nhất 1 mart')

Coverage check (13/13 bảng raw → marts):
  products        → ['Mart 1 (direct join)', 'Mart 3 (master)', 'Mart 5 (indirect via order_items)']
  customers       → ['Mart 1 (direct join)', 'Mart 2 (direct join)']
  promotions      → ['Mart 1 (direct join)', 'Mart 5 (master)']
  geography       → ['Mart 1 (direct join)', 'Mart 2 (direct join)']
  orders          → ['Mart 1 (direct join)', 'Mart 2 (master)', 'Mart 4 (agg daily)']
  order_items     → ['Mart 1 (master)', 'Mart 2 (agg)', 'Mart 3 (agg)', 'Mart 5 (agg)']
  payments        → ['Mart 2 (direct join)']
  shipments       → ['Mart 2 (direct join)']
  returns         → ['Mart 2 (agg)', 'Mart 3 (agg)', 'Mart 4 (agg daily)']
  reviews         → ['Mart 2 (agg)', 'Mart 3 (agg)']
  sales_train     → ['Mart 4 (master)']
  inventory       → ['Mart 3 (agg lifetime)', 'Mart 4 (broadcast monthly)']
  web_traffic     → ['Mart 4 (direct)']

✓ Tất cả 13 bảng đều xuất hiện trong ít nhất 1 mart


## ✅ Done!

**Phân vai team từ đây**:
- **DA Phần 2 EDA**:
  - Customer behavior, geo analysis: dùng `mart2_orders`
  - Product diagnostics, return analysis: dùng `mart3_products`
  - Cross-dimension EDA (region × segment × promo × age): dùng `mart1_order_lines`
  - Promotion ROI: dùng `mart5_promotion_perf`
  - Temporal trends: dùng `mart4_daily_ops`

- **MLE Phần 3 Forecasting**:
  - Backbone forecasting: `mart4_daily_ops` (đã có cả train+test period + date_dim)
  - ⚠️ Lưu ý leakage: từ 2023-01-01 trở đi, các cột non-date đều NaN → chỉ dùng date_dim features cho test period

**Quan trọng cho DA**:
- Filter `flag_order_before_signup == 0` nếu phân tích customer tenure
- Filter `is_revenue_eligible == 1` khi tính tổng doanh thu thật
- `flag_status_shipment_mismatch == 1` là 564 đơn anomaly